In [1]:
import pandas as pd, os, sqlite3
print(os.getcwd())
print(os.listdir('../data'))

c:\Projects\collections-project\notebooks
['POS_CASH_balance.csv', 'previous_application.csv']


In [2]:
pd.read_csv('../data/POS_CASH_balance.csv', nrows=100000).head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [3]:
pos = pd.read_csv('../data/POS_CASH_balance.csv',
    usecols=['SK_ID_PREV','SK_ID_CURR','MONTHS_BALANCE','SK_DPD','NAME_CONTRACT_STATUS'],
    dtype={'SK_ID_PREV':'int32','SK_ID_CURR':'int32','MONTHS_BALANCE':'int16','SK_DPD':'int32'})
print(pos.shape)

(10001358, 5)


In [4]:
print(pos['NAME_CONTRACT_STATUS'].value_counts())
print(pos['SK_DPD'].describe())

NAME_CONTRACT_STATUS
Active                   9151119
Completed                 744883
Signed                     87260
Demand                      7065
Returned to the store       5461
Approved                    4917
Amortized debt               636
Canceled                      15
XNA                            2
Name: count, dtype: int64
count    1.000136e+07
mean     1.160693e+01
std      1.327140e+02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      4.231000e+03
Name: SK_DPD, dtype: float64


In [5]:
loan = pos['SK_ID_PREV'].iloc[0]
pos[pos['SK_ID_PREV'] == loan].sort_values('MONTHS_BALANCE')

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,NAME_CONTRACT_STATUS,SK_DPD
2862153,1803195,182943,-34,Active,0
6477531,1803195,182943,-33,Active,0
2865318,1803195,182943,-32,Active,0
0,1803195,182943,-31,Active,0
4794783,1803195,182943,-30,Active,0
8125053,1803195,182943,-29,Active,0
3337839,1803195,182943,-28,Active,0
5564316,1803195,182943,-27,Active,0
5628351,1803195,182943,-26,Active,0
3093748,1803195,182943,-25,Active,0


In [6]:
prev = pd.read_csv('../data/previous_application.csv',
    usecols=['SK_ID_PREV','SK_ID_CURR','NAME_CONTRACT_TYPE','AMT_CREDIT'])
print(prev.shape, prev['SK_ID_PREV'].is_unique)

(1670214, 4) True


In [7]:
ids = pos['SK_ID_CURR'].drop_duplicates().sample(30000, random_state=42)
pos  = pos[pos['SK_ID_CURR'].isin(ids)]
prev = prev[prev['SK_ID_CURR'].isin(ids)]
print(pos.shape, prev.shape)

(897388, 5) (148470, 4)


In [8]:
print(pos.isna().sum())
print(pos.duplicated(['SK_ID_PREV','MONTHS_BALANCE']).sum())
print((pos['SK_DPD'] < 0).sum())
print(prev['AMT_CREDIT'].isna().sum(), (prev['AMT_CREDIT'] <= 0).sum())

SK_ID_PREV              0
SK_ID_CURR              0
MONTHS_BALANCE          0
NAME_CONTRACT_STATUS    0
SK_DPD                  0
dtype: int64
0
0
0 29583


In [9]:
pos  = pos.drop_duplicates(['SK_ID_PREV','MONTHS_BALANCE'])
prev = prev[prev['AMT_CREDIT'] > 0].dropna(subset=['AMT_CREDIT'])
prev = prev.drop_duplicates('SK_ID_PREV')

In [10]:
conn = sqlite3.connect('../collections.db')
pos.to_sql('pos', conn, if_exists='replace', index=False)
prev.to_sql('prev', conn, if_exists='replace', index=False)
print(pd.read_sql('SELECT COUNT(*) AS n FROM pos', conn))

        n
0  897388
